# Flask — Rotas, Templates e Métodos HTTP

Este notebook apresenta uma mini API em Flask com exemplos de:

- Rotas simples com `GET`
- Retorno em JSON com `jsonify`
- Página HTML usando `render_template`
- Cadastro com `POST`
- Atualização completa com `PUT`
- Atualização parcial com `PATCH`
- Remoção com `DELETE`
- Testes usando `requests`

> Observação: no Jupyter Notebook, a célula com `app.run()` fica executando continuamente. Para testar as requisições no mesmo notebook, rode o servidor em uma célula e use outro notebook, outro terminal, ou pare o servidor antes de executar outras células. Mais abaixo há uma versão usando `threading` apenas para facilitar testes em aula.


## 1. Instalação

Caso o Flask ainda não esteja instalado:


In [1]:
# !pip install flask requests

## 2. Estrutura básica da aplicação

Vamos criar uma API simples de filmes. A lista abaixo funciona como um "banco de dados" temporário em memória.

Isso significa que, quando o servidor for reiniciado, os dados voltam ao estado inicial.


In [2]:
from flask import Flask, jsonify, request, render_template
import os

app = Flask(__name__)

In [3]:
filmes = [
    {'id': 1, 'titulo': 'Interstellar', 'ano': 2014, 'genero': 'Ficção científica'},
    {'id': 2, 'titulo': 'Parasita', 'ano': 2019, 'genero': 'Drama'},
    {'id': 3, 'titulo': 'Whiplash', 'ano': 2014, 'genero': 'Drama'},
]

## 3. Criando uma página HTML com `render_template`

O Flask procura arquivos HTML dentro de uma pasta chamada `templates`.

Vamos criar essa pasta e um arquivo `home.html`.


In [4]:
os.makedirs('templates', exist_ok=True)

html_home = '''
<!DOCTYPE html>
<html>

<head>
    <meta charset="UTF-8">
    <title>API de Filmes</title>

    <style>
        body {
            font-family: Arial, sans-serif;
            background-color: #f5f5f5;
            text-align: center;
            margin-top: 60px;
        }

        a {
            display: block;
            margin: 12px;
        }
    </style>

</head>

<body>

    <h1>API de Filmes</h1>

    <p>Servidor funcionando.</p>

    <hr width="300">

    <a href="/filmes">Ver todos os filmes</a>

    <a href="/filmes/1">Buscar filme de ID 1</a>

    <a href="/filmes/busca?nome=parasita">
        Buscar filme por nome
    </a>

</body>

</html>
'''

with open('templates/template1.html', 'w', encoding='utf-8') as arquivo:
    arquivo.write(html_home)

print('Arquivo templates/template1.html criado com sucesso!')

Arquivo templates/template1.html criado com sucesso!


In [5]:
@app.route('/')
def home():
    return render_template('template1.html')

## 4. Rotas GET

O método `GET` é usado para consultar dados.

Exemplos:

```text
GET /filmes
GET /filmes/1
GET /filmes/busca?nome=parasita
```


In [6]:
@app.route('/filmes', methods=['GET'])
def listar_filmes():
    return jsonify(filmes)


@app.route('/filmes/<int:id>', methods=['GET'])
def buscar_filme_por_id(id):
    for filme in filmes:
        if filme['id'] == id:
            return jsonify(filme)

    return jsonify({'erro': 'Filme não encontrado'}), 404


@app.route('/filmes/busca', methods=['GET'])
def buscar_filme_por_nome():
    nome = request.args.get('nome')

    if not nome:
        return jsonify({'erro': 'Informe o parâmetro nome'}), 400

    for filme in filmes:
        if filme['titulo'].lower() == nome.lower():
            return jsonify(filme)

    return jsonify({'erro': 'Filme não encontrado'}), 404

## 5. Rota POST — criar novo filme

O método `POST` é usado para cadastrar um novo recurso.

Aqui, vamos enviar um JSON com os dados do novo filme.


In [7]:
@app.route('/filmes', methods=['POST'])
def criar_filme():
    novo_filme = request.get_json()

    if not novo_filme:
        return jsonify({'erro': 'Nenhum dado enviado'}), 400

    campos_obrigatorios = ['titulo', 'ano', 'genero']

    for campo in campos_obrigatorios:
        if campo not in novo_filme:
            return jsonify({'erro': f'O campo {campo} é obrigatório'}), 400

    novo_id = max([filme['id'] for filme in filmes]) + 1

    filme = {
        'id': novo_id,
        'titulo': novo_filme['titulo'],
        'ano': novo_filme['ano'],
        'genero': novo_filme['genero']
    }

    filmes.append(filme)

    return jsonify({
        'mensagem': 'Filme criado com sucesso',
        'filme': filme
    }), 201

## 6. Rota PUT — atualizar filme inteiro

O método `PUT` normalmente substitui o recurso inteiro.

Neste exemplo, para atualizar um filme com `PUT`, o aluno precisa enviar todos os campos principais: `titulo`, `ano` e `genero`.


In [8]:
@app.route('/filmes/<int:id>', methods=['PUT'])
def atualizar_filme_completo(id):
    dados = request.get_json()

    if not dados:
        return jsonify({'erro': 'Nenhum dado enviado'}), 400

    campos_obrigatorios = ['titulo', 'ano', 'genero']

    for campo in campos_obrigatorios:
        if campo not in dados:
            return jsonify({'erro': f'O campo {campo} é obrigatório para PUT'}), 400

    for filme in filmes:
        if filme['id'] == id:
            filme['titulo'] = dados['titulo']
            filme['ano'] = dados['ano']
            filme['genero'] = dados['genero']

            return jsonify({
                'mensagem': 'Filme atualizado com sucesso',
                'filme': filme
            })

    return jsonify({'erro': 'Filme não encontrado'}), 404

## 7. Rota PATCH — atualizar parte do filme

O método `PATCH` é usado para atualização parcial.

Por exemplo, posso alterar apenas o `genero` ou apenas o `ano`, sem reenviar o filme inteiro.


In [9]:
@app.route('/filmes/<int:id>', methods=['PATCH'])
def atualizar_filme_parcial(id):
    dados = request.get_json()

    if not dados:
        return jsonify({'erro': 'Nenhum dado enviado'}), 400

    for filme in filmes:
        if filme['id'] == id:
            if 'titulo' in dados:
                filme['titulo'] = dados['titulo']

            if 'ano' in dados:
                filme['ano'] = dados['ano']

            if 'genero' in dados:
                filme['genero'] = dados['genero']

            return jsonify({
                'mensagem': 'Filme atualizado parcialmente',
                'filme': filme
            })

    return jsonify({'erro': 'Filme não encontrado'}), 404

## 8. Rota DELETE — remover filme

O método `DELETE` remove um recurso.


In [10]:
@app.route('/filmes/<int:id>', methods=['DELETE'])
def deletar_filme(id):
    for filme in filmes:
        if filme['id'] == id:
            filmes.remove(filme)

            return jsonify({
                'mensagem': 'Filme removido com sucesso',
                'filme_removido': filme
            })

    return jsonify({'erro': 'Filme não encontrado'}), 404

## 9. Rodando o servidor

### Opção A — jeito mais simples

Rode a célula abaixo. Ela vai ficar executando enquanto o servidor estiver ativo.

```python
app.run(debug=False, port=5002)
```

Para parar o servidor, interrompa a execução da célula.


In [ ]:
app.run(debug=False, port=5003)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5003
Press CTRL+C to quit
127.0.0.1 - - [09/Jun/2026 20:06:14] "GET / HTTP/1.1" 200 -


## 10. Rodando o servidor com `threading` para testar no mesmo notebook

Esta opção é útil em aula, porque permite que o servidor fique rodando e o aluno consiga executar as células de teste com `requests` logo abaixo.

Para uma aula introdutória, você pode comentar que isso é apenas uma conveniência do Jupyter.


In [13]:
from threading import Thread

def rodar_servidor():
    app.run(debug=False, port=5002, use_reloader=False)

# Execute apenas uma vez.
servidor = Thread(target=rodar_servidor)
servidor.start()

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5002
Press CTRL+C to quit
127.0.0.1 - - [09/Jun/2026 19:51:45] "GET / HTTP/1.1" 200 -


## 11. Testando as rotas com `requests`

Agora vamos testar cada tipo de requisição.

> Importante: antes de executar estas células, o servidor Flask precisa estar rodando.


In [14]:
import requests

base_url = 'http://127.0.0.1:5002'

### 11.1 Testando a página inicial HTML

In [34]:
headers = {'Accept': 'application/json'}

resposta = requests.get(
    f'{base_url}/',
    headers=headers,
    timeout=5
)

print('Status:', resposta.status_code)
print('URL final:', resposta.url)
print('Encoding:', resposta.encoding)
print('Content-Type:', resposta.headers.get('Content-Type'))
print('Servidor:', resposta.headers.get('Server'))
print('Tamanho (bytes):', len(resposta.content))
print('Tempo:', resposta.elapsed.total_seconds(), 's')

127.0.0.1 - - [09/Jun/2026 19:55:58] "GET / HTTP/1.1" 200 -


Status: 200
URL final: http://127.0.0.1:5002/
Encoding: utf-8
Content-Type: text/html; charset=utf-8
Servidor: Werkzeug/2.2.3 Python/3.11.7
Servidor: None
Tamanho (bytes): 1301
Tempo: 0.003035 s


In [30]:
print(resposta.text)


<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>API de Filmes</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            background: linear-gradient(120deg, #1f1f1f, #444);
            color: white;
            text-align: center;
            padding: 50px;
        }

        .card {
            background-color: rgba(255, 255, 255, 0.12);
            padding: 30px;
            border-radius: 16px;
            max-width: 600px;
            margin: auto;
        }

        a {
            display: block;
            color: #ffd166;
            margin: 12px;
            font-size: 18px;
            text-decoration: none;
        }

        a:hover {
            text-decoration: underline;
        }

        img {
            width: 220px;
            border-radius: 12px;
            margin-bottom: 20px;
        }
    </style>
</head>
<body>
    <div class="card">
        <img src="https://upload.wikimedia.org/wikipedia/commons/3/3c/Fla

### 11.2 GET — listar todos os filmes

In [16]:
resposta = requests.get(f'{base_url}/filmes')

print(resposta.status_code)
resposta.json()

127.0.0.1 - - [09/Jun/2026 19:52:08] "GET /filmes HTTP/1.1" 200 -


200


[{'ano': 2014,
  'genero': 'Ficção científica',
  'id': 1,
  'titulo': 'Interstellar'},
 {'ano': 2019, 'genero': 'Drama', 'id': 2, 'titulo': 'Parasita'},
 {'ano': 2014, 'genero': 'Drama', 'id': 3, 'titulo': 'Whiplash'}]

### 11.3 GET — buscar filme por ID

In [17]:
resposta = requests.get(f'{base_url}/filmes/1')

print(resposta.status_code)
resposta.json()

127.0.0.1 - - [09/Jun/2026 19:52:11] "GET /filmes/1 HTTP/1.1" 200 -


200


{'ano': 2014, 'genero': 'Ficção científica', 'id': 1, 'titulo': 'Interstellar'}

### 11.4 GET — buscar filme por nome usando query string

In [18]:
resposta = requests.get(f'{base_url}/filmes/busca?nome=Parasita')

print(resposta.status_code)
resposta.json()

127.0.0.1 - - [09/Jun/2026 19:52:12] "GET /filmes/busca?nome=Parasita HTTP/1.1" 200 -


200


{'ano': 2019, 'genero': 'Drama', 'id': 2, 'titulo': 'Parasita'}

Também podemos passar os parâmetros usando o argumento `params` do `requests`:

In [19]:
resposta = requests.get(
    f'{base_url}/filmes/busca',
    params={'nome': 'Whiplash'}
)

print(resposta.status_code)
resposta.json()

127.0.0.1 - - [09/Jun/2026 19:52:13] "GET /filmes/busca?nome=Whiplash HTTP/1.1" 200 -


200


{'ano': 2014, 'genero': 'Drama', 'id': 3, 'titulo': 'Whiplash'}

### 11.5 POST — cadastrar novo filme

In [20]:
novo_filme = {
    'titulo': 'Cidade de Deus',
    'ano': 2002,
    'genero': 'Drama'
}

resposta = requests.post(
    f'{base_url}/filmes',
    json=novo_filme
)

print(resposta.status_code)
resposta.json()

127.0.0.1 - - [09/Jun/2026 19:52:17] "POST /filmes HTTP/1.1" 201 -


201


{'filme': {'ano': 2002,
  'genero': 'Drama',
  'id': 4,
  'titulo': 'Cidade de Deus'},
 'mensagem': 'Filme criado com sucesso'}

Conferindo se o filme foi adicionado:

In [21]:
resposta = requests.get(f'{base_url}/filmes')

print(resposta.status_code)
resposta.json()

127.0.0.1 - - [09/Jun/2026 19:52:19] "GET /filmes HTTP/1.1" 200 -


200


[{'ano': 2014,
  'genero': 'Ficção científica',
  'id': 1,
  'titulo': 'Interstellar'},
 {'ano': 2019, 'genero': 'Drama', 'id': 2, 'titulo': 'Parasita'},
 {'ano': 2014, 'genero': 'Drama', 'id': 3, 'titulo': 'Whiplash'},
 {'ano': 2002, 'genero': 'Drama', 'id': 4, 'titulo': 'Cidade de Deus'}]

### 11.6 PUT — atualizar o filme inteiro

In [22]:
filme_atualizado = {
    'titulo': 'Cidade de Deus',
    'ano': 2002,
    'genero': 'Crime/Drama'
}

resposta = requests.put(
    f'{base_url}/filmes/4',
    json=filme_atualizado
)

print(resposta.status_code)
resposta.json()

127.0.0.1 - - [09/Jun/2026 19:52:24] "PUT /filmes/4 HTTP/1.1" 200 -


200


{'filme': {'ano': 2002,
  'genero': 'Crime/Drama',
  'id': 4,
  'titulo': 'Cidade de Deus'},
 'mensagem': 'Filme atualizado com sucesso'}

### 11.7 PATCH — atualizar apenas um campo

In [23]:
atualizacao_parcial = {
    'genero': 'Drama brasileiro'
}

resposta = requests.patch(
    f'{base_url}/filmes/4',
    json=atualizacao_parcial
)

print(resposta.status_code)
resposta.json()

127.0.0.1 - - [09/Jun/2026 19:52:26] "PATCH /filmes/4 HTTP/1.1" 200 -


200


{'filme': {'ano': 2002,
  'genero': 'Drama brasileiro',
  'id': 4,
  'titulo': 'Cidade de Deus'},
 'mensagem': 'Filme atualizado parcialmente'}

### 11.8 DELETE — remover um filme

In [24]:
resposta = requests.delete(f'{base_url}/filmes/4')

print(resposta.status_code)
resposta.json()

127.0.0.1 - - [09/Jun/2026 19:52:27] "DELETE /filmes/4 HTTP/1.1" 200 -


200


{'filme_removido': {'ano': 2002,
  'genero': 'Drama brasileiro',
  'id': 4,
  'titulo': 'Cidade de Deus'},
 'mensagem': 'Filme removido com sucesso'}

Conferindo a lista final:

In [25]:
resposta = requests.get(f'{base_url}/filmes')

print(resposta.status_code)
resposta.json()

127.0.0.1 - - [09/Jun/2026 19:52:28] "GET /filmes HTTP/1.1" 200 -


200


[{'ano': 2014,
  'genero': 'Ficção científica',
  'id': 1,
  'titulo': 'Interstellar'},
 {'ano': 2019, 'genero': 'Drama', 'id': 2, 'titulo': 'Parasita'},
 {'ano': 2014, 'genero': 'Drama', 'id': 3, 'titulo': 'Whiplash'}]

## 12. Resumo dos métodos HTTP

| Método | Uso principal | Exemplo |
|---|---|---|
| GET | Consultar dados | `GET /filmes` |
| POST | Criar novo recurso | `POST /filmes` |
| PUT | Atualizar recurso inteiro | `PUT /filmes/1` |
| PATCH | Atualizar parte do recurso | `PATCH /filmes/1` |
| DELETE | Remover recurso | `DELETE /filmes/1` |

Uma API simples de filmes poderia seguir este padrão:

```text
GET     /filmes
GET     /filmes/1
GET     /filmes/busca?nome=Parasita
POST    /filmes
PUT     /filmes/1
PATCH   /filmes/1
DELETE  /filmes/1
```
